## config

In [7]:
from app.config import validate_env, ensure_dirs
validate_env()
ensure_dirs()
print("config OK")


config OK


## Loader



In [1]:
from app.ingestion.loader import load_pdf
docs = load_pdf("data/documents/rag_survey.pdf")
print(len(docs), docs[0].metadata)

67 {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': 'rag_survey.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1', 'file_path': 'data\\documents\\rag_survey.pdf', 'doc_id': 'e1f208bba353', 'page_number': 1}


## Chunker
semantic ko chhod kar, kyunki wo Cohere ko bahut baar call karta hai

In [2]:
from app.ingestion.loader import load_pdf
from app.ingestion.chunker import create_chunks

docs = load_pdf("data/documents/rag_survey.pdf")

for strategy in ["recursive", "fixed", "structure_aware"]:
    chunks = create_chunks(docs, strategy)
    ids = [c.metadata["chunk_id"] for c in chunks]
    print(strategy, len(chunks), ids == list(range(len(chunks))))
    print(chunks[0].metadata)

sections = {c.metadata.get("section") for c in create_chunks(docs, "structure_aware")}
print(len(sections), list(sections)[:8])

recursive 234 True
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': 'rag_survey.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1', 'file_path': 'data\\documents\\rag_survey.pdf', 'doc_id': 'e1f208bba353', 'page_number': 1, 'start_index': 0, 'chunk_id': 0, 'chunk_uid': 'e1f208bba353:recursive:0', 'chunk_strategy': 'recursive'}
fixed 238 True
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'crea

## Embeddings(service.py)

In [3]:
from app.embeddings.service import embed_query_text, embed_texts

v1 = embed_query_text("What is RAG?")
v2 = embed_query_text("What is RAG?")
print(len(v1), v1[:3] == v2[:3])

print(len(embed_texts(["hello", "world"])))

try:
    embed_texts(["ok", "   "])
except ValueError as e:
    print(e)

1024 True
2
Cannot embed empty text at position 1.


# qdrant store 


In [4]:
import logging
logging.basicConfig(level=logging.INFO)

from app.ingestion.loader import load_pdf
from app.ingestion.chunker import create_chunks
from app.embeddings.service import embed_chunks
from app.vectorstore.qdrant_store import (
    create_qdrant_client, recreate_collection, store_chunks, get_collection_stats,
)

docs = load_pdf("data/documents/rag_survey.pdf")
chunks = create_chunks(docs, "recursive")
embeddings = embed_chunks(chunks)

client = create_qdrant_client()
recreate_collection(client)
store_chunks(client, chunks, embeddings)
store_chunks(client, chunks, embeddings)
print(get_collection_stats(client))
client.close()

INFO:app.ingestion.loader:Loaded 67 pages from rag_survey.pdf
INFO:httpx:HTTP Request: POST https://api.cohere.com/v1/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cohere.com/v1/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cohere.com/v1/embed "HTTP/1.1 200 OK"
INFO:app.embeddings.service:Embedded 234 texts in 4.29s
INFO:app.vectorstore.qdrant_store:Collection 'rag_documents' created.
INFO:app.vectorstore.qdrant_store:Stored 234 points in 'rag_documents'.
INFO:app.vectorstore.qdrant_store:Stored 234 points in 'rag_documents'.


{'name': 'rag_documents', 'points_count': 234}


## vectorstore/registry.py

In [6]:
from app.vectorstore.registry import register_collection, list_registered, resolve_collection

register_collection("rag_recursive_1000_150", "recursive", 1000, 150, "abc123", "rag_survey.pdf", 234)
print(list_registered())
print(resolve_collection("default"))

KeyError: 'collections'

## ingestion/ingest.py

## retrieval/retriever

In [8]:
from app.retrieval.retriever import retrieve_chunks, build_context

res = retrieve_chunks("What is Retrieval-Augmented Generation?", top_k=5)
for r in res:
    print(r.vector_rank, r.chunk_id, r.page_number, round(r.score, 3), r.preview(80))

print(build_context(res)[:400])

INFO:httpx:HTTP Request: POST https://api.cohere.com/v1/embed "HTTP/1.1 200 OK"
INFO:app.retrieval.retriever:Vector search in 'rag_documents': 5 results in 4 ms


1 0 1 0.633 Retrieval-Augmented Generation for Natural Language Processing: A Survey Shangyu...
2 11 4 0.594 • Text Classification • Dialogue System • A Taxonomy of RAG Evaluation • Key Eva...
3 218 62 0.586 11:1–17 Spotify (2017) Annoy. Software, URL https://github.com/spotify/annoy Spr...
4 10 4 0.566 𝑧𝑧1, 𝑧𝑧2, … , 𝑧𝑧𝑘𝑘 Text/Feature concatenation 𝑥𝑥 = [𝑞𝑞⨁ 𝑧𝑧1⨁𝑧𝑧2⨁𝑧𝑧3⨁ … ⨁𝑧𝑧𝑘𝑘] Ge...
5 14 5 0.563 neighbor (ANN) search, and a vector database for storing external knowledge in t...
[Source 1]
File: rag_survey.pdf
Page: 1
Content:
Retrieval-Augmented Generation for Natural
Language Processing: A Survey
Shangyu Wu1,2, Ying Xiong 2†, Yufei Cui 3, Haolun Wu 3, Can
Chen3, Ye Yuan 3, Lianming Huang 1, Xue Liu 2, Tei-Wei Kuo 4,
Nan Guan 1, Chun Jason Xue 2
1City University of Hong Kong.
2Mohamed bin Zayed University of Artificial Intelligence.
3McGill University, Mila.
4National Ta


In [ ]:
from app.retrieval.retriever import retrieve_chunks
res = retrieve_chunks("What is Retrieval-Augmented Generation?", top_k=5)
print(res[0].text)


##  retrieval/bm25.py

In [ ]:
from app.retrieval.bm25 import retrieve_bm25

for q in ["Annoy approximate nearest neighbor", "What is Retrieval-Augmented Generation?"]:
    print("\nQ:", q)
    for r in retrieve_bm25(q, top_k=5):
        print(r.bm25_rank, r.chunk_id, r.page_number, round(r.bm25_score, 2), r.preview(70))

## retrieval/hybrid.py

In [ ]:
from app.retrieval.retriever import retrieve_chunks
from app.retrieval.bm25 import retrieve_bm25
from app.retrieval.hybrid import retrieve_hybrid, overlap_stats

q = "What are the major types of retrieval fusion?"

v = retrieve_chunks(q, top_k=30)
b = retrieve_bm25(q, top_k=30)
print("Overlap:", overlap_stats(v, b))

for method in ["rrf", "weighted"]:
    print("\n", method)
    for r in retrieve_hybrid(q, top_k=5, method=method):
        print(r.hybrid_rank, r.chunk_id, r.page_number,
              "vec#", r.vector_rank, "bm25#", r.bm25_rank,
              round(r.hybrid_score, 2), r.preview(60))

## retrieval/reranker.py

Check karne ke liye: Phase 2 ka poora test (4 stages ek sawal pe

In [ ]:
from app.retrieval.retriever import retrieve_chunks
from app.retrieval.bm25 import retrieve_bm25
from app.retrieval.hybrid import retrieve_hybrid
from app.retrieval.reranker import retrieve_reranked, rank_changes

q = "What is the main challenge in the retriever module?"

print("VECTOR")
for r in retrieve_chunks(q, top_k=5):
    print(r.vector_rank, r.chunk_id, r.page_number, round(r.vector_score, 3), r.preview(60))

print("\nBM25")
for r in retrieve_bm25(q, top_k=5):
    print(r.bm25_rank, r.chunk_id, r.page_number, round(r.bm25_score, 2), r.preview(60))

print("\nHYBRID")
for r in retrieve_hybrid(q, top_k=5):
    print(r.hybrid_rank, r.chunk_id, r.page_number, round(r.hybrid_score, 2), r.preview(60))

print("\nRERANKED")
final = retrieve_reranked(q, top_k=3)
for r in final:
    print(r.rerank_rank, r.chunk_id, r.page_number, round(r.rerank_score, 2), r.preview(60))
print(rank_changes(final))

## generation/llm.py

In [ ]:
from app.retrieval.retriever import retrieve_chunks, build_context
from app.generation.llm import create_llm, generate_answer

llm = create_llm()

# 1. jawab milna chahiye
q = "What is Retrieval-Augmented Generation?"
ctx = build_context(retrieve_chunks(q, top_k=5))
r = generate_answer(llm, q, ctx)
print(r["answer"])
print(r["refused"], r["latency_ms"], "ms", r["usage"], r["model"])

# 2. mana karna chahiye (document mein ye nahi hai)
q2 = "Who is the current Prime Minister of India?"
ctx2 = build_context(retrieve_chunks(q2, top_k=5))
r2 = generate_answer(llm, q2, ctx2)
print(r2["answer"], r2["refused"])

## retrieval/advanced/query_transformer.py

In [ ]:
from app.retrieval.advanced.query_transformer import rewrite_query

history = [
    {"role": "user", "content": "What is Retrieval-Augmented Generation?"},
    {"role": "assistant", "content": "It combines retrieval of external knowledge with an LLM."},
]

r = rewrite_query("What are its main challenges?", history)
print(r["query"], "|", r["changed"], r["fallback_reason"], r["latency_ms"], "ms")

r2 = rewrite_query("What is RAG?")   # bina history
print(r2["query"], r2["llm_called"], r2["fallback_reason"])

## query_expansion.py aur multi_query.py

In [ ]:
from app.retrieval.advanced.query_expansion import expand_query_detailed
from app.retrieval.advanced.multi_query import generate_query_variants, retrieve_multi_query

q = "What are the major types of retrieval fusion?"

e = expand_query_detailed(q)
print("EXPANDED:", e["query"], "| terms:", e["added_terms"], e["fallback_reason"])

info = generate_query_variants(q, num_queries=3)
print("\nQUERIES:")
for text in info["queries"]:
    print(" -", text)

print("\nFUSED RESULTS:")
for r in retrieve_multi_query(q, top_k=5, variants=info["queries"]):
    print(r.hybrid_rank, r.chunk_id, r.page_number, round(r.hybrid_score, 2), r.preview(60))

## retrieval/advanced/hyde.py

In [ ]:
from app.retrieval.advanced.hyde import create_hyde_vector, retrieve_hyde
from app.retrieval.retriever import retrieve_chunks

q = "What is the main challenge in the retriever module?"

h = create_hyde_vector(q)
print("PASSAGE:", h["hypothetical_document"])
print(h["vector_dim"], h["embed_as"], h["fallback_reason"], h["latency_ms"], "ms", h["usage"])

print("\nPLAIN VECTOR SEARCH")
for r in retrieve_chunks(q, top_k=5):
    print(r.vector_rank, r.chunk_id, r.page_number, round(r.score, 3), r.preview(60))

print("\nHYDE SEARCH")
for r in retrieve_hyde(q, top_k=5, hyde=h):
    print(r.vector_rank, r.chunk_id, r.page_number, round(r.score, 3), r.preview(60))

## crag.py aur context_compressor.py

In [ ]:
from app.retrieval.hybrid import retrieve_hybrid
from app.retrieval.advanced.crag import run_crag
from app.retrieval.advanced.context_compressor import compress_context_detailed

questions = [
    "What is the main challenge in the retriever module?",   # answerable
    "Who is the current Prime Minister of India?",           # unanswerable
]

for q in questions:
    chunks = retrieve_hybrid(q, top_k=5)
    out = run_crag(q, chunks, top_k=5)
    print("\nQ:", q)
    print("action:", out["action"], "| llm_calls:", out["llm_calls"])
    for g in out["grades"]:
        print("  grade:", g["verdict"], "-", g["reason"])

    c = compress_context_detailed(q, out["chunks"], min_chars=0)
    print("  compression ratio:", c["ratio"], [p["status"] for p in c["per_chunk"]], c["fallback_reason"])

## pipeline_config.py aur tracing/trace.py

Step 19: PipelineConfig

In [ ]:
from app.pipeline_config import PipelineConfig, get_preset

PipelineConfig()                              # baseline: vector search, seedha LLM
get_preset("hybrid_rerank", top_k=3)          # ready-made experiment
PipelineConfig(retrieval_method="hybrid", use_reranker=True, use_crag=True)

Step 20: Trace (record book)

In [ ]:
trace = Trace.new(question, config)

with trace.stage("retrieve", method="hybrid") as stage:
    chunks = retrieve_hybrid(...)
    stage.details["count"] = len(chunks)      # time apne aap naap jaata hai

Ab tak ka picture: Phase 4 mein ye sab kaise judega
PipelineConfig  ──►  pipeline.py  ──►  Trace (har stage ka record)
 (kya chalana hai)   (chalata hai)         │
                                           ▼
                                      trace_store.py  ──►  dashboard / eval
                                      (disk pe save)
Check karne ke liye (30 second, koi API nahi chahiye)

In [ ]:
from app.pipeline_config import get_preset, PipelineConfig
from app.tracing.trace import Trace

cfg = get_preset("hybrid_rerank", top_k=3)
print(cfg.label(), cfg.fingerprint())
print(cfg.diff_from(PipelineConfig()))

t = Trace.new("What is RAG?", cfg)
with t.stage("retrieve", method="hybrid") as s:
    s.details["count"] = 5
print(t.finish().to_json()[:500])

21. tracing/trace_store.py

In [ ]:
from app.pipeline_config import get_preset
from app.tracing.trace import Trace
from app.tracing.trace_store import get_store, save_trace

cfg = get_preset("hybrid_rerank")
t = Trace.new("What is RAG?", cfg)
with t.stage("retrieve") as s:
    s.details["count"] = 5
t.answer, t.action = "RAG combines retrieval and generation.", "answered"
t.finish()

print(save_trace(t, run_id="demo"))                 # True
store = get_store()
print(store.list(limit=3))
print(store.stats(run_id="demo"))
print(store.get(t.trace_id).stages[0].name)         # retrieve
print(store.path)                                   # data/traces/traces.db

Step 22: app/pipeline.py

In [ ]:
run_pipeline("What is RAG?")                                   # baseline (vector search)
run_pipeline("What is RAG?", "hybrid_rerank")                  # preset ka naam
run_pipeline("What is RAG?", {"retrieval_method": "bm25", "top_k": 3})   # dict
run_pipeline("What is RAG?", PipelineConfig(use_crag=True))    # object

In [ ]:
from app.pipeline import run_pipeline

for preset in ["vector", "hybrid", "hybrid_rerank"]:
    t = run_pipeline("What is the main challenge in the retriever module?", preset, run_id="smoke")
    print("\n", t.config_label, "->", t.action, "|", t.total_latency_ms, "ms |", t.usage["total_tokens"], "tokens")
    for s in t.stages:
        print("  ", s.name.ljust(16), s.status.ljust(8), s.latency_ms, "ms", s.error or "")
    print("  ANSWER:", t.answer[:150])

23. rag_service.py (Phase 4)

In [ ]:
sources = build_sources(results)          # (file, page) pe DEDUPLICATE hua, list chhoti
for source, result in zip(sources, results):    # results lambi, sources chhoti
    source["chunk_id"] = str(result.id)
    source["preview"] = payload["text"][:250]

In [ ]:
from app.services.rag_service import answer_question

r = answer_question("What is Retrieval-Augmented Generation?")
print(r["answer"])
print(r["sources"][:2])
print(r["trace_id"], r["config_label"], r["latency_ms"], "ms", r["usage"])

r2 = answer_question("Who is the current Prime Minister of India?")
print(r2["refused"], "|", r2["answer"], "|", r2["sources"])

In [1]:
from app.evaluation.dataset import load_dataset

ds = load_dataset("evaluation/dataset.json")
print(ds.stats())
q = ds.get(7)
print(q.question, "|", q.ground_truth, "|", [(e.page, e.snippet[:50]) for e in q.evidence])
print(len(ds.filter(expected="unanswerable")), "unanswerable")

{'questions': 8, 'answerable': 5, 'unanswerable': 3, 'reviewed': 0, 'by_type': {'single_hop': 5, 'unanswerable': 3}, 'by_tag': {}, 'evidence_snippets': 0}
What is the capital of France? | None | []
3 unanswerable
